# 06 · Feature importance — CPU models (Logistic Regression, Random Forest)

Step 06 explains the models trained in step 04, in three notebooks:
- **this one** (CPU): Logistic Regression and Random Forest, 36 triples;
- `06-importance-gpu.ipynb` (GPU): XGBoost, MLP, GNNs, and GNNExplainer;
- `06-importance-plots.ipynb` (CPU, after the other two): tables and plots.

**Runtime: CPU.** Expected time: a few hours, almost all for the Random Forest (500 deep trees): about 2,600 predictions of the test for the permutation importance of each seed, plus TreeSHAP. A timing cell (section 4) measures the speed first, chooses the number of licit rows for TreeSHAP and prints the estimate. Runs over several Colab sessions resume where they stopped.

**The two questions.** Step 05 compared menus: models **retrained** on different groups of features. This step looks **inside** the trained models of step 04 (no new training) and asks, for every triple (menu, model, seed): which columns and which **families** of columns does the model use, and in which direction?

**Families.** The columns of a menu belong to six families, as decided in step 02: LF (93 Local features of Elliptic), AF (72 Aggregate features), A (6 degrees and counts), B (11 centralities), C (7 motifs and curvature), D (9 economic columns of Elliptic++). A menu with two families or more gets one value per family; `local` (LF only) and `econ` (D only) do not, because shuffling their only family would shuffle the whole menu.

**1 · Permutation importance** (Breiman, 2001; Fisher, Rudin and Dominici, 2019), for every model. The values of one column, or of all the columns of a family together (the same permutation for all of them, so a transaction keeps its own combination), are shuffled among the test transactions; the model is **not** retrained; the importance is the drop of the test PR-AUC:
$$I_j = \mathrm{AP}(\text{test}) - \frac{1}{R}\sum_{r=1}^{R}\mathrm{AP}(\text{test with column } j \text{ shuffled by permutation } r), \qquad R = 5.$$
- The values are shuffled **within each time step**: a transaction of time step 42 never gets a value of time step 47, so the drop measures only "the model uses this column", not "the value comes from another period".
- The 5 permutations come from fixed seeds (0–4) and are **the same for every model** (one set for the evaluation rows, used by LogReg, RF, XGBoost and MLP; one set for all the nodes of the test graph, used by the GNNs, whose neighbours are shuffled too).
- On the validation (time steps 31–40) the same is done **for the families only**, to see whether what a model relies on changes between the two periods.
- The drop of the ROC-AUC is saved too.

**2 · Attributions** $\phi_{ij}$: how much column $j$ pushes transaction $i$ towards "illicit". For every model the same definition (SHAP values, Lundberg and Lee, 2017), computed with the exact method of the model:
- **Logistic Regression** — exact formula, in log-odds: $\phi_{ij} = w_j\,(x_{ij} - \bar x_j)$, with $\bar x$ the mean of the train rows; base value $b + w\cdot\bar x$, and base + $\sum_j \phi_{ij}$ = the logit.
- **Random Forest** — TreeSHAP (Lundberg et al., 2020), exact for trees, in probability; on the explain sample (below), because the forest has deep trees.
- **XGBoost** — TreeSHAP of XGBoost itself (`pred_contribs`), in log-odds, on the whole test.
- **MLP and GNNs** — **Integrated Gradients** (Sundararajan, Taly and Yan, 2017), in logits: with the baseline $x'$ = the mean of the train,
$$\phi_{ij} = (x_{ij} - x'_j)\,\frac{1}{m}\sum_{k=1}^{m} \frac{\partial f}{\partial x_j}\Big(x' + \tfrac{k - 1/2}{m}(x_i - x')\Big), \qquad m = 50,$$
and $\sum_j \phi_{ij} = f(x_i) - f(x')$ (completeness, checked). For a linear model it gives exactly the formula of the Logistic Regression. For a GNN the logit of a node depends also on the features of its neighbours: the gradient is taken with respect to the features of **every** node of its neighbourhood, so each $\phi_{ij}$ splits into the part of the node itself and the part of its neighbours.

Per column and per family (sum of the columns of the family, $\Phi_{iF} = \sum_{j\in F}\phi_{ij}$): the mean of $|\phi|$ over the test, the mean over the illicit transactions, the share of the total, the **density** = share / share of the columns ($> 1$: the family weighs more than its number of columns), the **direction** (Spearman correlation between the value of a column and its $\phi$: $> 0$ means high values push towards "illicit") and, for the GNNs, the **share of the neighbours**.

**The explain sample** (one for every model, in `explain_sample.csv`): all 524 illicit transactions of the test plus 1,000 licit ones drawn in proportion to the time steps (seed 0). Each row has a weight, the number of test transactions it stands for (illicit 1, licit 9,342 / 1,000 = 9.342), so weighted means over the sample estimate means over the test. Inside it, the 80 nodes of the GNNExplainer core (60 illicit in proportion to the time steps, 20 licit).

| File (in `06-feature-importance/`) | Content |
|---|---|
| `explain_sample.csv` | the explain sample: `row`, `txId`, time step, label, weight, order of the licit rows, GNNExplainer core |
| `permutation/<triple>.csv` | one row per column and per family: data (test or validation), mean and standard deviation over the 5 shuffles of the drop of PR-AUC and ROC-AUC |
| `shap_summary/<triple>.csv` | one row per column, per family and for the whole menu: method, mean $|\phi|$, mean $|\phi|$ on the illicit, mean $\phi$, share, density, direction |
| `shap/<triple>.parquet` | seed 0 only: the matrix $\phi$ (rows × columns), with `row`, label, weight, base value and output |
| `checks.csv` | every check: reloaded models give the saved scores, additivity / completeness |
| `done/<triple>.json` | times and settings; written last: its presence means "triple finished" |
| `logs/06-importance-cpu.log` | the log lines |

**Resume.** A triple is skipped when its `done` file exists. If Colab disconnects, run the notebook again from the top: at most the triple that was running is lost. Files are written through a temporary file and renamed.

**Shared code.** The cells `SHARED CODE` are identical copies of the cells of steps 03 and 04 (data, transformations, models): the models are rebuilt exactly as they were trained. The notebook stops if the files of step 04 were written by another version of that code. The cell `IMPORTANCE CODE` is identical in `06-importance-cpu` and `06-importance-gpu`.

## 0. Setup

In [1]:
# Install the libraries that Colab does not have by default (only if they are missing).
import importlib.util
import subprocess
import sys

for package, module in [('shap', 'shap'), ('xgboost', 'xgboost')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

In [2]:
# Connect Google Drive: every input and output of the project lives in DATA_PATH.
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DATA_PATH = Path('/content/drive/MyDrive/UNIVERSITA/block-crypto/data')
INPUT_DIR = DATA_PATH / '02-feature-menus'          # output of step 02
RUNS_DIR = DATA_PATH / '04-final-runs'              # output of step 04: models, predictions, summaries
OUT_DIR = DATA_PATH / '06-feature-importance'       # output of step 06
LOCAL_INPUT = Path('/content/input')                # local copy of the input files

Mounted at /content/drive


In [3]:
# Copy the inputs from Drive to the Colab disk: reading a local file is much faster.
import shutil

LOCAL_INPUT.mkdir(exist_ok=True)
for name in ['config.json', 'dataset.parquet', 'edges.parquet']:
    source, target = INPUT_DIR / name, LOCAL_INPUT / name
    assert source.exists(), f'{source} is missing: run step 02 first'
    if not target.exists() or target.stat().st_size != source.stat().st_size:
        shutil.copy(source, target)
    print(f'{name}: {target.stat().st_size / 1e6:,.1f} MB')

config.json: 0.1 MB
dataset.parquet: 76.1 MB
edges.parquet: 2.2 MB


In [4]:
NOTEBOOK = 'cpu'
MODELS = ['logreg', 'rf']      # the models explained by this notebook
ONLY_PAIRS = None              # None = every pair; or a list such as ['transparent__rf'] to run only some pairs
SETTINGS = {
    'sample': {'seed': 0, 'licit': 1000, 'core_illicit': 60, 'core_licit': 20},   # the explain sample (same in both notebooks)
    'shuffles': 5,             # permutations per column (seeds 0-4)
    'rf_shap_seeds': 'auto',   # seeds of the RF explained with TreeSHAP: 'auto' = [0, 1, 2] if the budget allows, else [0]
    'rf_licit': 'auto',        # licit rows of the sample for TreeSHAP of the RF: 'auto' = 1,000, or fewer to fit the budget (>= 200)
    'rf_shap_budget_hours': 2.0,
    'matrix_seed': 0,          # the seed whose full matrix of attributions is saved
    'ig_steps': 50, 'ig_gnn_seeds': [],   # not used here (GPU notebook)
}
FORCE_RECOMPUTE = False        # True = delete the results of this notebook's models and compute them again

## 1. Shared code

Data, seeds and metrics (1/4) and classical models (2/4): identical to the tuning and final-run notebooks. Then the importance code, identical in the two computing notebooks of this step.

In [5]:
# ==== SHARED CODE 1/4 · data, seeds, metrics · SHARED_CODE_VERSION 1 ====
# Identical copies in: 03-tuning-cpu, 03-tuning-gpu, and later 04-final-runs-cpu, 04-final-runs-gpu.
# Change every copy together and raise SHARED_CODE_VERSION.
import json
import os
import random
import sys
import time
from datetime import datetime, timezone
from types import SimpleNamespace

os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')   # deterministic cuBLAS: must be set before PyTorch uses the GPU
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

SHARED_CODE_VERSION = 1
CONFIG_VERSION = 1                 # the version of config.json this code was written for
DETERMINISTIC_ALGORITHMS = True    # ask PyTorch for deterministic GPU kernels (only a warning where none exists)


def set_seed(seed):
    """Fix every random generator: Python, NumPy and, when it is loaded, PyTorch (CPU, every GPU, cuDNN)."""
    random.seed(seed)
    np.random.seed(seed)
    if 'torch' in sys.modules:
        import torch
        torch.manual_seed(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        torch.use_deterministic_algorithms(DETERMINISTIC_ALGORITHMS, warn_only=True)


def atomic(path, write):
    """Write through a temporary file, then rename it: a run that stops while writing
    never leaves a half-written file on Drive."""
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    write(tmp)
    os.replace(tmp, path)


def to_builtin(value):
    """Make NumPy numbers JSON-serializable."""
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, np.bool_):
        return bool(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    raise TypeError(f'not JSON-serializable: {type(value)}')


def write_csv(df, path):
    atomic(path, lambda tmp: df.to_csv(tmp, index=False, float_format='%.10g'))


def write_json(obj, path):
    text = json.dumps(obj, indent=1, default=to_builtin, ensure_ascii=False)
    atomic(path, lambda tmp: tmp.write_text(text, encoding='utf-8'))


def fmt_time(seconds):
    """Format a duration as '12.3s', '4m 05s' or '1h 07m'."""
    if seconds < 60:
        return f'{seconds:.1f}s'
    minutes, seconds = divmod(int(seconds), 60)
    if minutes < 60:
        return f'{minutes}m {seconds:02d}s'
    hours, minutes = divmod(minutes, 60)
    return f'{hours}h {minutes:02d}m'


def load_inputs(folder):
    """config.json, dataset.parquet and edges.parquet of step 02, with the arrays every later step needs."""
    config = json.loads((folder / 'config.json').read_text(encoding='utf-8'))
    assert config['version'] == CONFIG_VERSION, f"config.json has version {config['version']}, this code expects {CONFIG_VERSION}"
    dataset = pd.read_parquet(folder / 'dataset.parquet')
    edges = pd.read_parquet(folder / 'edges.parquet')
    assert len(dataset) == config['files']['n_rows'] and len(edges) == config['files']['n_edges']
    y = dataset['y'].to_numpy(dtype=np.float64, na_value=np.nan)
    return SimpleNamespace(
        config=config, dataset=dataset,
        split=dataset['split'].to_numpy(), in_eval=dataset['in_eval'].to_numpy(dtype=bool), illicit=(y == 1),
        src=edges['src_row'].to_numpy(np.int64), dst=edges['dst_row'].to_numpy(np.int64))


def scale(values, spec):
    """One column of the profile `scaled` (config['transforms']): z = (t - mean) / std, with
    t = sign(x) log(1 + |x|) for the two log transformations and t = x otherwise (binary: mean 0, std 1)."""
    x = np.asarray(values, dtype=np.float64)
    if spec['transform'] in ('log1p+z', 'signed_log1p+z'):
        x = np.sign(x) * np.log1p(np.abs(x))
    return (x - spec['mean']) / spec['std']


def feature_matrix(data, menu, profile):
    """The input matrix of a menu for every node (float32): raw values (profile `raw`) or transformed
    (profile `scaled`). The menu `none` gives the constant input x = 1."""
    columns = data.config['menus'][menu]['columns']
    if not columns:
        return np.ones((len(data.dataset), 1), dtype=np.float32)
    X = data.dataset[columns].to_numpy(np.float64)
    if profile == 'scaled':
        specs = data.config['transforms']['profiles']['scaled']['columns']
        for k, col in enumerate(columns):
            X[:, k] = scale(X[:, k], specs[col])
    return X.astype(np.float32)


def eval_rows(data, split):
    """Row numbers of the evaluation rows of a split (labeled and with the Elliptic++ data)."""
    return np.flatnonzero((data.split == split) & data.in_eval)


def suggest_params(trial, space):
    """Ask an Optuna trial for one value of every hyper-parameter of a search space of config.json."""
    params = {}
    for name, spec in space.items():
        if spec['type'] == 'categorical':
            params[name] = trial.suggest_categorical(name, spec['choices'])
        elif spec['type'] == 'float':
            params[name] = trial.suggest_float(name, spec['low'], spec['high'], log=spec['log'])
        elif spec['type'] == 'int':
            params[name] = trial.suggest_int(name, spec['low'], spec['high'], log=spec['log'])
        else:
            raise ValueError(f'unknown type {spec["type"]} for {name}')
    return params

In [6]:
# ==== SHARED CODE 2/4 · classical models · SHARED_CODE_VERSION 1 ====
# Identical copies in: 03-tuning-cpu, 03-tuning-gpu, and later 04-final-runs-cpu, 04-final-runs-gpu.
# Change every copy together and raise SHARED_CODE_VERSION.
import warnings

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

# XGBoost on the GPU with NumPy inputs prints this at every prediction; it is harmless.
warnings.filterwarnings('ignore', message='.*Falling back to prediction using DMatrix.*')


def fit_classical(model_name, params, fixed, X_train, y_train, X_val, y_val, seed, final=False):
    """Train Logistic Regression, Random Forest or XGBoost on the train rows.
    XGBoost stops adding trees when the validation PR-AUC stops improving (early stopping).
    `final=True` (step 04) also computes the out-of-bag score of the Random Forest."""
    fixed = dict(fixed)
    info = {}
    start = time.perf_counter()
    if model_name == 'logreg':
        model = LogisticRegression(**fixed, **params, random_state=seed)
        model.fit(X_train, y_train)
    elif model_name == 'rf':
        oob = bool(fixed.pop('oob_score_in_final_runs', False)) and final
        model = RandomForestClassifier(**fixed, **params, oob_score=oob, random_state=seed)
        model.fit(X_train, y_train)
        if oob:
            info['oob_average_precision'] = float(average_precision_score(
                y_train, np.nan_to_num(model.oob_decision_function_[:, 1])))
    elif model_name == 'xgb':
        model = XGBClassifier(**fixed, **params, random_state=seed)
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
        info['best_iteration'] = int(model.best_iteration)
    else:
        raise ValueError(model_name)
    info['seconds'] = time.perf_counter() - start
    scores = model.predict_proba(X_val)[:, 1]
    return SimpleNamespace(model=model, val_scores=scores, val_ap=float(average_precision_score(y_val, scores)), info=info)

In [7]:
# ==== IMPORTANCE CODE · explain sample, shuffles, permutation, attributions, files · identical in 06-importance-cpu and 06-importance-gpu ====
# Change both copies together. It uses the shared code above (blocks 1 and 2, and block 3 for the networks).
# PyTorch and SHAP are imported only inside the functions that need them.
import gc
import math

import joblib
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score

SPLITS = ('train', 'val', 'test')
FAMILIES = ['LF', 'AF', 'A', 'B', 'C', 'D']
GNN_MODELS = ('sage', 'gatv2', 'cheb', 'gin')


def triple_id(menu, model, seed):
    return f'{menu}__{model}__seed{seed}'


def free_memory():
    gc.collect()
    if 'torch' in sys.modules:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def sigmoid64(z):
    """The score in float64: the float32 sigmoid would round every logit above about 17 to exactly 1 (ties)."""
    with np.errstate(over='ignore'):
        return 1 / (1 + np.exp(-np.asarray(z, dtype=np.float64)))


# ---------- families ----------
def column_family(config):
    """Every kept column -> its family (LF, AF, A, B, C, D), as decided in step 02."""
    return {col: fam for fam in FAMILIES for col in config['columns']['families'][fam]['kept']}


def menu_units(config, menu, families_only=False):
    """What is shuffled together: every column alone and, if the menu has two families or more, every family
    (a menu with one family gets no family value: shuffling the family would shuffle the whole menu).
    Returns {name: (level, family, positions of the columns)} and {family: positions}."""
    columns = config['menus'][menu]['columns']
    fam = column_family(config)
    groups = {}
    for k, col in enumerate(columns):
        groups.setdefault(fam[col], []).append(k)
    units = {} if families_only else {col: ('column', fam[col], [k]) for k, col in enumerate(columns)}
    if len(groups) > 1:
        units.update({f: ('family', f, ks) for f, ks in groups.items()})
    return units, groups


# ---------- the explain sample ----------
def largest_remainder(total, counts):
    """Split `total` into integers proportional to `counts` (largest remainders; ties go to the first)."""
    counts = np.asarray(counts, dtype=np.float64)
    quota = total * counts / counts.sum()
    base = np.floor(quota).astype(int)
    order = np.argsort(-(quota - base), kind='stable')
    base[order[:total - base.sum()]] += 1
    return base


def draw_by_time_step(rng, rows, steps_of_rows, total):
    """`total` of `rows` drawn without replacement, in proportion to the time steps."""
    steps = np.unique(steps_of_rows)
    counts = largest_remainder(total, [(steps_of_rows == t).sum() for t in steps])
    picked = [rng.choice(rows[steps_of_rows == t], size=k, replace=False) for t, k in zip(steps, counts) if k]
    return np.sort(np.concatenate(picked)) if picked else np.array([], dtype=np.int64)


def build_explain_sample(data, settings):
    """One explain sample for every model: every illicit transaction of the test plus `licit` licit ones drawn in
    proportion to the time steps. weight = how many test transactions a row stands for (illicit 1, licit
    9,342 / 1,000), so weighted means over the sample estimate means over the test. Inside it, the core of
    GNNExplainer: `core_illicit` illicit and `core_licit` licit rows, also in proportion to the time steps.
    `licit_order` is a fixed random order of the licit rows: a smaller subset takes the first ones."""
    rng = np.random.default_rng(settings['seed'])
    rows = eval_rows(data, 'test')
    step = data.dataset['time_step'].to_numpy()
    illicit = data.illicit[rows]
    licit_rows = draw_by_time_step(rng, rows[~illicit], step[rows[~illicit]], settings['licit'])
    illicit_rows = rows[illicit]
    core = np.concatenate([draw_by_time_step(rng, illicit_rows, step[illicit_rows], settings['core_illicit']),
                           draw_by_time_step(rng, licit_rows, step[licit_rows], settings['core_licit'])])
    sample = np.concatenate([illicit_rows, licit_rows])
    order = np.full(len(sample), -1)
    order[len(illicit_rows):] = rng.permutation(len(licit_rows))
    n_licit_test = int((~illicit).sum())
    table = pd.DataFrame({'row': sample, 'txId': data.dataset['txId'].to_numpy()[sample], 'time_step': step[sample],
                          'y': data.illicit[sample].astype(int),
                          'weight': np.where(data.illicit[sample], 1.0, n_licit_test / len(licit_rows)),
                          'licit_order': order, 'gnnexplainer_core': np.isin(sample, core)})
    return table.sort_values('row', kind='stable').reset_index(drop=True)


def load_or_build_sample(data, settings, path):
    """The first notebook that runs writes the sample; the other one rebuilds it and checks that it is identical."""
    table = build_explain_sample(data, settings)
    assert abs(table['weight'].sum() - len(eval_rows(data, 'test'))) < 1e-6, 'the weights must add up to the test size'
    if path.exists():
        saved = pd.read_csv(path)
        same = (np.array_equal(saved['row'], table['row']) and np.allclose(saved['weight'], table['weight'])
                and np.array_equal(saved['licit_order'], table['licit_order'])
                and np.array_equal(saved['gnnexplainer_core'].astype(bool), table['gnnexplainer_core']))
        assert same, f'{path.name} differs from the sample rebuilt here: different settings or data'
    else:
        write_csv(table, path)
    return table


def sample_subset(sample, n_licit):
    """All illicit rows and the first `n_licit` licit rows of the fixed random order, re-weighted to stand for the test."""
    keep = (sample['y'] == 1) | ((sample['licit_order'] >= 0) & (sample['licit_order'] < n_licit))
    sub = sample[keep].copy()
    sub.loc[sub['y'] == 0, 'weight'] = sample.loc[sample['y'] == 0, 'weight'].sum() / n_licit
    return sub.reset_index(drop=True)


# ---------- permutation importance ----------
def within_step_permutations(steps_of_positions, n_shuffles, first_seed=0):
    """One permutation of the positions per shuffle r (seed first_seed + r): values move only between positions of the
    same time step. The same seeds and the same positions give the same permutations to every model."""
    perms = []
    for r in range(n_shuffles):
        rng = np.random.default_rng(first_seed + r)
        perm = np.arange(len(steps_of_positions))
        for t in np.unique(steps_of_positions):
            idx = np.flatnonzero(steps_of_positions == t)
            perm[idx] = idx[rng.permutation(len(idx))]
        perms.append(perm)
    return perms


def permutation_importance(score, X, y, units, perms):
    """Drop of PR-AUC and ROC-AUC when the columns of a unit are shuffled and the model is NOT retrained.
    The columns of a unit (one column, or a whole family) move with the same permutation, so a row keeps its own
    combination of their values. `score(X)` gives the scores of the evaluation rows; X is a NumPy array or a PyTorch
    tensor, changed in place and restored after every unit."""
    base = score(X)
    base_ap, base_roc = average_precision_score(y, base), roc_auc_score(y, base)
    is_tensor = not isinstance(X, np.ndarray)
    if is_tensor:
        import torch
        perms = [torch.as_tensor(np.asarray(p, dtype=np.int64), device=X.device) for p in perms]
    rows = []
    for name, (level, family, positions) in units.items():
        cols = torch.as_tensor(positions, dtype=torch.long, device=X.device) if is_tensor else np.asarray(positions)
        original = X[:, cols].clone() if is_tensor else X[:, cols].copy()
        ap, roc = [], []
        for perm in perms:
            X[:, cols] = original[perm]
            s = score(X)
            ap.append(base_ap - average_precision_score(y, s))
            roc.append(base_roc - roc_auc_score(y, s))
        X[:, cols] = original
        spread = (lambda v: float(np.std(v, ddof=1))) if len(perms) > 1 else (lambda v: 0.0)
        rows.append({'level': level, 'name': name, 'family': family, 'n_columns': len(positions),
                     'ap_drop': float(np.mean(ap)), 'ap_drop_std': spread(ap),
                     'roc_drop': float(np.mean(roc)), 'roc_drop_std': spread(roc),
                     'base_ap': base_ap, 'base_roc': base_roc, 'shuffles': len(perms)})
    restored = score(X)
    return pd.DataFrame(rows), float(np.max(np.abs(restored - base)))


# ---------- attributions ----------
def attribution_summary(phi, values, y, weights, columns, families, groups, neighbours=None):
    """Summary of a matrix of attributions phi (rows x columns) over rows that stand for the test with `weights`.
    For every column, every family (menus with two families or more) and the whole menu: weighted mean of |phi|
    (the test), mean |phi| over the illicit rows, weighted mean of phi, share of the total, density
    (share / share of the columns), and for the columns the direction: Spearman correlation between the value of
    the column and its phi (> 0: high values push towards "illicit"). `neighbours` (GNNs): the part of phi that comes
    from the features of the neighbours; neighbour_share = mean |neighbour part| / (mean |own part| + mean |neighbour part|)."""
    w = np.asarray(weights, dtype=np.float64)
    w = w / w.sum()
    illicit = np.asarray(y).astype(bool)
    n = len(columns)

    def one(p, nb=None):
        out = {'mean_abs': float(w @ np.abs(p)),
               'mean_abs_illicit': float(np.abs(p[illicit]).mean()) if illicit.any() else np.nan,
               'mean': float(w @ p)}
        if nb is not None:
            own_abs, nb_abs = float(w @ np.abs(p - nb)), float(w @ np.abs(nb))
            out['neighbour_share'] = nb_abs / (own_abs + nb_abs) if own_abs + nb_abs > 0 else np.nan
        return out

    rows = []
    for j, col in enumerate(columns):
        row = {'level': 'column', 'name': col, 'family': families[j], 'n_columns': 1,
               **one(phi[:, j], None if neighbours is None else neighbours[:, j])}
        x, p = values[:, j], phi[:, j]
        ok = np.isfinite(x)
        row['direction'] = (float(spearmanr(x[ok], p[ok]).correlation)
                            if ok.sum() > 2 and np.ptp(x[ok]) > 0 and np.ptp(p[ok]) > 0 else np.nan)
        rows.append(row)
    total = sum(r['mean_abs'] for r in rows)
    for r in rows:
        r['share'] = r['mean_abs'] / total if total > 0 else np.nan
        r['density'] = r['share'] * n
    if len(groups) > 1:
        family_rows = []
        for fam, positions in groups.items():
            family_rows.append({'level': 'family', 'name': fam, 'family': fam, 'n_columns': len(positions),
                                **one(phi[:, positions].sum(axis=1),
                                      None if neighbours is None else neighbours[:, positions].sum(axis=1)),
                                'direction': np.nan})
        family_total = sum(r['mean_abs'] for r in family_rows)
        for r in family_rows:
            r['share'] = r['mean_abs'] / family_total if family_total > 0 else np.nan
            r['density'] = r['share'] / (r['n_columns'] / n)
        rows += family_rows
    rows.append({'level': 'menu', 'name': 'all', 'family': 'all', 'n_columns': n,
                 **one(phi.sum(axis=1), None if neighbours is None else neighbours.sum(axis=1)),
                 'direction': np.nan, 'share': 1.0, 'density': 1.0})
    return pd.DataFrame(rows)


def linear_shap(model, X, mean):
    """SHAP of a logistic regression in log-odds (exact for a linear model, features taken as independent):
    phi_j = w_j (x_j - mean_j); base = b + w . mean; base + sum_j phi_j = the logit."""
    w = model.coef_[0].astype(np.float64)
    phi = (X.astype(np.float64) - mean) * w
    return phi, float(model.intercept_[0] + w @ mean)


def tree_shap_forest(model, X):
    """TreeSHAP (exact, path-dependent) of a scikit-learn Random Forest, in probability of "illicit"."""
    import shap
    explainer = shap.TreeExplainer(model)
    values = explainer.shap_values(X, check_additivity=False)
    if isinstance(values, list):
        values = values[1]
    elif np.ndim(values) == 3:
        values = values[:, :, 1]
    base = np.ravel(explainer.expected_value)
    return np.asarray(values, dtype=np.float64), float(base[1] if len(base) > 1 else base[0])


def integrated_gradients_rows(net, x, baseline, steps, chunk=8192):
    """Integrated Gradients of the logit of every row of an MLP (Sundararajan et al., 2017), with the midpoint rule:
    phi_j = (x_j - x'_j) * (1/m) sum_k d f(x' + a_k (x - x')) / d x_j, with a_k = (k + 1/2) / m.
    Completeness: sum_j phi_j = f(x) - f(x'), up to the error of the rule (measured and returned)."""
    import torch
    net.eval()
    out, errors, logits = [], [], []
    for start in range(0, x.shape[0], chunk):
        xc = x[start:start + chunk]
        total = torch.zeros_like(xc)
        for k in range(steps):
            point = (baseline + (k + 0.5) / steps * (xc - baseline)).detach().requires_grad_(True)
            grad, = torch.autograd.grad(net(point).sum(), point)
            total += grad
        attr = ((xc - baseline) * total / steps).double()
        with torch.no_grad():
            f_x = net(xc).double()
            f_base = net(baseline.expand_as(xc)).double()
        gap = (f_x - f_base)
        errors.append(((attr.sum(dim=1) - gap).abs() / gap.abs().clamp(min=1e-6)).cpu().numpy())
        out.append(attr.cpu().numpy())
        logits.append(f_x.cpu().numpy())
    return np.vstack(out), float(f_base[0]), np.concatenate(logits), np.concatenate(errors)


def receptive_hops(model, params):
    """How far a GNN looks: one hop per layer, K - 1 hops per layer for ChebNet (polynomials up to order K - 1)."""
    return params['layers'] * (params['K'] - 1) if model == 'cheb' else params['layers']


def gnn_integrated_gradients(net, part, targets, baseline, hops, steps, max_nodes=400_000):
    """Integrated Gradients of the logit of every target node with respect to the features of EVERY node of its
    neighbourhood. The neighbourhood is the subgraph within hops + 1 hops (the extra hop keeps the degree of the outer
    nodes, which ChebNet's normalisation uses); the logit on it must equal the logit on the whole graph. The m points of
    the path are m copies of the subgraph in one disjoint batch. For each target: the attribution of its own features
    (own) and the sum over the other nodes (neighbours), both of length d; total = own + neighbours."""
    import torch
    from torch_geometric.utils import k_hop_subgraph
    net.eval()
    with torch.no_grad():
        full = net(part.x, part.edge_index).double().cpu().numpy()
    n_all, d = part.x.shape
    own, nbr, err, sub_gap, sizes = [], [], [], [], []
    for t in targets:
        subset, ei, mapping, _ = k_hop_subgraph(int(t), hops + 1, part.edge_index, relabel_nodes=True, num_nodes=n_all)
        x_sub = part.x[subset]
        n, centre = len(subset), int(mapping[0])
        grads = torch.zeros_like(x_sub)
        per_batch = max(1, min(steps, max_nodes // max(n, 1)))
        for start in range(0, steps, per_batch):
            m = min(per_batch, steps - start)
            alphas = (torch.arange(start, start + m, device=x_sub.device, dtype=x_sub.dtype) + 0.5) / steps
            points = (baseline + alphas[:, None, None] * (x_sub - baseline)[None]).reshape(m * n, d)
            points = points.detach().requires_grad_(True)
            batch_edges = torch.cat([ei + j * n for j in range(m)], dim=1)
            out = net(points, batch_edges).view(m, n)[:, centre].sum()
            grad, = torch.autograd.grad(out, points)
            grads += grad.view(m, n, d).sum(dim=0)
        attr = ((x_sub - baseline) * grads / steps).double()
        with torch.no_grad():
            logit = float(net(x_sub, ei)[centre])
            logit_base = float(net(baseline.expand_as(x_sub).contiguous(), ei)[centre])
        a_own = attr[centre].cpu().numpy()
        a_all = attr.sum(dim=0).cpu().numpy()
        own.append(a_own)
        nbr.append(a_all - a_own)
        gap = logit - logit_base
        err.append(abs(a_all.sum() - gap) / max(abs(gap), 1e-6))
        sub_gap.append(abs(logit - full[t]))
        sizes.append(n)
    own, nbr = np.vstack(own), np.vstack(nbr)
    return SimpleNamespace(total=own + nbr, neighbours=nbr, completeness=np.array(err),
                           subgraph_difference=np.array(sub_gap), sizes=np.array(sizes), logits=full[np.asarray(targets)])


# ---------- models of step 04 ----------
def load_classical(model, path, xgb_device='cuda'):
    if model in ('logreg', 'rf'):
        return joblib.load(path)
    m = XGBClassifier()
    m.load_model(str(path))
    m.get_booster().set_param({'device': xgb_device})
    return m


def classical_score(model, m, record):
    """The score function of a classical model, in float64. XGBoost uses the trees up to its best iteration."""
    if model == 'xgb':
        booster, best = m.get_booster(), (0, int(record['training']['best_iteration']) + 1)
        return lambda X: sigmoid64(booster.inplace_predict(X, iteration_range=best, predict_type='margin'))
    return lambda X: m.predict_proba(X)[:, 1].astype(np.float64)


def load_network(model, record, path):
    import torch
    net = build_neural(model, record['params'], record['d_in'], record['fixed'])
    net.load_state_dict(torch.load(path, map_location=DEVICE))
    net.eval()
    return net


def network_score(net, edge_index, mask):
    import torch

    def score(x):
        with torch.no_grad():
            return sigmoid64(net(x, edge_index)[mask].double().cpu().numpy())
    return score


# ---------- files, checks, progress ----------
class Progress:
    """Log lines with triples done and left and an estimate of the time left (expected work of each triple,
    corrected by the speed measured in this session). Lines are also appended to a log file."""

    def __init__(self, expected, done, log_path):
        self.expected, self.done, self.log_path = expected, set(done), log_path
        self.work, self.start, self.lines = 0.0, time.time(), []

    def finish(self, key):
        self.done.add(key)
        self.work += self.expected[key]

    def status(self):
        elapsed = time.time() - self.start
        speed = elapsed / self.work if self.work > 0 else None
        left = sum(s for t, s in self.expected.items() if t not in self.done)
        eta = fmt_time(left * speed) if speed else 'measuring'
        return f'done {len(self.done)}, left {len(self.expected) - len(self.done)} · elapsed {fmt_time(elapsed)}, left ~{eta}'

    def print(self, line):
        print(line, flush=True)
        self.lines.append(f'{datetime.now():%Y-%m-%d %H:%M:%S} {line}')
        self.flush()

    def flush(self):
        if self.lines and self.log_path is not None:
            self.log_path.parent.mkdir(parents=True, exist_ok=True)
            with open(self.log_path, 'a', encoding='utf-8') as f:
                f.write('\n'.join(self.lines) + '\n')
        self.lines = []


def add_checks(path, rows):
    """Append rows (triple, check, value, limit, ok) to checks.csv, replacing older rows of the same check."""
    old = pd.read_csv(path) if path.exists() else pd.DataFrame(columns=['triple', 'check', 'value', 'limit', 'ok'])
    new = pd.DataFrame(rows)
    keys = set(zip(new['triple'], new['check']))
    old = old[[(t, c) not in keys for t, c in zip(old['triple'], old['check'])]]
    write_csv(pd.concat([old, new], ignore_index=True), path)


def expected_work(config, menu, model, settings):
    """Relative work of one triple, for the estimate of the time left (the speed is measured while running)."""
    n = len(config['menus'][menu]['columns'])
    units, groups = menu_units(config, menu)
    n_fam = len(groups) if len(groups) > 1 else 0
    predicts = (len(units) + n_fam) * settings['shuffles']
    per_predict = {'logreg': 0.01, 'rf': 1.0, 'xgb': 0.03, 'mlp': 0.01, 'sage': 0.05, 'gatv2': 0.1, 'cheb': 0.08,
                   'gin': 0.05}[model]
    licit = settings['rf_licit'] if isinstance(settings['rf_licit'], int) else 1000
    attribution = {'logreg': 0.1, 'rf': 0.004 * n * (524 + licit), 'xgb': 1, 'mlp': 1}.get(
        model, 1524 * 0.02 * max(n, 1) / 50)
    return 1 + predicts * per_predict + attribution


def explain_triple(ctx, menu, model, seed):
    """Every measure of one triple, then its files. done/<triple>.json is written last: "triple finished"."""
    data, config, dirs, settings = ctx.data, ctx.data.config, ctx.dirs, ctx.settings
    tid = triple_id(menu, model, seed)
    record = ctx.records[(menu, model, seed)]
    start = time.perf_counter()
    columns = config['menus'][menu]['columns']
    fam = column_family(config)
    families = [fam[c] for c in columns]
    units, groups = menu_units(config, menu)
    family_units, _ = menu_units(config, menu, families_only=True)
    step = data.dataset['time_step'].to_numpy()
    rows = {split: eval_rows(data, split) for split in SPLITS}
    y = {split: data.illicit[rows[split]].astype(int) for split in SPLITS}
    saved = pd.read_parquet(dirs.runs / 'predictions' / f'{tid}.parquet')
    saved = saved[saved['split'] == 'test']
    assert np.array_equal(saved['row'].to_numpy(), rows['test']), 'the saved predictions are in another order'
    X_all = feature_matrix(data, menu, config['models'][model]['profile'])   # the menu none gives x = 1
    values_of = lambda used: data.dataset.iloc[used][columns].to_numpy(np.float64)   # raw values, for the direction
    model_path = dirs.runs / record['files']['model']
    checks, timings, tables = [], {}, []
    method, matrix = None, None

    def check(name, value, limit):
        checks.append({'triple': tid, 'check': name, 'value': float(value), 'limit': limit, 'ok': bool(value <= limit)})
        assert value <= limit, f'{tid}: {name} = {value:.3g} > {limit:g}'

    if model in ('logreg', 'rf', 'xgb'):
        m = load_classical(model, model_path, settings.get('xgb_device', 'cuda'))
        score = classical_score(model, m, record)
        X = {split: X_all[rows[split]].copy() for split in ('test', 'val')}
        test_scores = score(X['test'])
        tolerance = 1e-6 if model == 'xgb' else 1e-12
        check('reload: largest score difference', np.max(np.abs(test_scores - saved['score'].to_numpy())), tolerance)
        t0 = time.perf_counter()
        perm_rows = {split: within_step_permutations(step[rows[split]], settings['shuffles']) for split in ('test', 'val')}
        table, restored = permutation_importance(score, X['test'], y['test'], units, perm_rows['test'])
        tables.append(table.assign(data='test'))
        if family_units:
            table, _ = permutation_importance(score, X['val'], y['val'], family_units, perm_rows['val'])
            tables.append(table.assign(data='val'))
        check('permutation: scores restored', restored, 1e-9)
        timings['permutation'] = time.perf_counter() - t0
        t0 = time.perf_counter()
        if model == 'logreg':
            mean = X_all[rows['train']].astype(np.float64).mean(axis=0)
            phi, base = linear_shap(m, X['test'], mean)
            output = base + phi.sum(axis=1)
            logit = m.decision_function(X['test']).astype(np.float64)
            check('linear SHAP: base + sum of phi = logit', np.max(np.abs(output - logit)), 1e-6)
            used, weights, method, space = rows['test'], np.ones(len(rows['test'])), 'linear_shap', 'log-odds'
        elif model == 'xgb':
            import xgboost
            booster, best = m.get_booster(), (0, int(record['training']['best_iteration']) + 1)
            contribs = booster.predict(xgboost.DMatrix(X['test']), pred_contribs=True, iteration_range=best)
            phi, base = contribs[:, :-1].astype(np.float64), float(contribs[0, -1])
            margin = booster.inplace_predict(X['test'], iteration_range=best, predict_type='margin')
            check('TreeSHAP: base + sum of phi = margin', np.max(np.abs(contribs.sum(axis=1) - margin)), 1e-3)
            output = contribs.sum(axis=1)
            used, weights, method, space = rows['test'], np.ones(len(rows['test'])), 'tree_shap', 'log-odds'
        elif seed in settings['rf_shap_seeds']:
            sub = sample_subset(ctx.sample, settings['rf_licit'])
            Xs = X_all[sub['row'].to_numpy()]
            phi, base = tree_shap_forest(m, Xs)
            output = base + phi.sum(axis=1)
            check('TreeSHAP: base + sum of phi = probability', np.max(np.abs(output - score(Xs))), 1e-4)
            used, weights, method, space = sub['row'].to_numpy(), sub['weight'].to_numpy(), 'tree_shap', 'probability'
        else:
            phi = None   # Random Forest on a seed without TreeSHAP (see the timing cell)
        timings['attribution'] = time.perf_counter() - t0
        summary = None
        if phi is not None:
            summary = attribution_summary(phi, values_of(used), data.illicit[used], weights, columns, families, groups)
            matrix = (used, weights, phi, None, base, output, space)
    else:
        import torch
        net = load_network(model, record, model_path)
        kind = 'rows' if model == 'mlp' else 'graph'
        part = {split: period_data(data, X_all, split, kind) for split in ('test', 'val')}
        nodes = {split: (rows[split] if kind == 'rows' else np.flatnonzero(data.split == split)) for split in ('test', 'val')}
        score = {split: network_score(net, part[split].edge_index, part[split].mask) for split in ('test', 'val')}
        test_scores = score['test'](part['test'].x)
        check('reload: largest score difference', np.max(np.abs(test_scores - saved['score'].to_numpy())), 1e-4)
        check('reload: test PR-AUC difference', abs(average_precision_score(y['test'], test_scores) -
                                                    average_precision_score(y['test'], saved['score'].to_numpy())), 1e-4)
        summary = None
        if columns:
            t0 = time.perf_counter()
            perms = {split: within_step_permutations(step[nodes[split]], settings['shuffles']) for split in ('test', 'val')}
            table, restored = permutation_importance(score['test'], part['test'].x, y['test'], units, perms['test'])
            tables.append(table.assign(data='test'))
            if family_units:
                table, _ = permutation_importance(score['val'], part['val'].x, y['val'], family_units, perms['val'])
                tables.append(table.assign(data='val'))
            check('permutation: scores restored', restored, 1e-5)
            timings['permutation'] = time.perf_counter() - t0
            t0 = time.perf_counter()
            if model == 'mlp':
                baseline = torch.tensor(X_all[rows['train']].astype(np.float64).mean(axis=0), dtype=torch.float32,
                                        device=DEVICE)
                phi, base, logits, completeness = integrated_gradients_rows(net, part['test'].x, baseline,
                                                                            settings['ig_steps'])
                check('Integrated Gradients: median completeness error', np.median(completeness), 0.05)
                used, weights, neighbours = rows['test'], np.ones(len(rows['test'])), None
                output = logits
            elif seed in settings['ig_gnn_seeds']:
                train_nodes = np.flatnonzero(data.split == 'train')
                baseline = torch.tensor(X_all[train_nodes].astype(np.float64).mean(axis=0), dtype=torch.float32,
                                        device=DEVICE)
                targets = np.searchsorted(nodes['test'], ctx.sample['row'].to_numpy())
                assert np.array_equal(nodes['test'][targets], ctx.sample['row'].to_numpy())
                ig = gnn_integrated_gradients(net, part['test'], targets, baseline,
                                              receptive_hops(model, record['params']), settings['ig_steps'])
                check('subgraph: largest logit difference from the whole graph', ig.subgraph_difference.max(), 1e-3)
                check('Integrated Gradients: median completeness error', np.median(ig.completeness), 0.05)
                checks.append({'triple': tid, 'check': 'Integrated Gradients: 95th percentile of the completeness error',
                               'value': float(np.percentile(ig.completeness, 95)), 'limit': np.nan, 'ok': True})
                phi, neighbours, output, base = ig.total, ig.neighbours, ig.logits, float('nan')
                used, weights = ctx.sample['row'].to_numpy(), ctx.sample['weight'].to_numpy()
                timings['subgraph_nodes_median'] = float(np.median(ig.sizes))
            else:
                phi = None
            if phi is not None:
                method = 'integrated_gradients'
                summary = attribution_summary(phi, values_of(used), data.illicit[used], weights, columns, families,
                                              groups, neighbours)
                matrix = (used, weights, phi, neighbours, base, output, 'logit')
            timings['attribution'] = time.perf_counter() - t0
        del part
    # ---- files ----
    if tables:
        write_csv(pd.concat(tables, ignore_index=True), dirs.out / 'permutation' / f'{tid}.csv')
    if summary is not None:
        write_csv(summary.assign(method=method, rows=len(matrix[0])), dirs.out / 'shap_summary' / f'{tid}.csv')
        if seed == settings['matrix_seed']:
            used, weights, phi, neighbours, base, output, space = matrix
            table = pd.DataFrame(phi, columns=columns)
            if neighbours is not None:
                table = pd.concat([table, pd.DataFrame(neighbours, columns=[f'{c}__neighbours' for c in columns])], axis=1)
            table.insert(0, 'row', used)
            table.insert(1, 'y', data.illicit[used].astype(int))
            table.insert(2, 'weight', weights)
            table.insert(3, 'base', base)
            table.insert(4, 'output', output)
            atomic(dirs.out / 'shap' / f'{tid}.parquet', lambda tmp: table.to_parquet(tmp, index=False))
    add_checks(dirs.out / 'checks.csv', checks)
    record_out = {'triple': tid, 'menu': menu, 'model': model, 'seed': seed, 'method': method,
                  'attribution_space': matrix[6] if matrix else None, 'timings': timings,
                  'seconds': time.perf_counter() - start, 'settings': {k: v for k, v in settings.items()},
                  'shared_code_version': SHARED_CODE_VERSION,
                  'created': datetime.now(timezone.utc).isoformat(timespec='seconds')}
    write_json(record_out, dirs.out / 'done' / f'{tid}.json')
    free_memory()
    return record_out


def plan_triples(ctx, models, only_pairs=None):
    """Every triple of this notebook, the summaries of step 04, and what is already done."""
    config = ctx.data.config
    pairs = [(menu, model) for menu, model in config['pairs']
             if model in models and (only_pairs is None or f'{menu}__{model}' in only_pairs)]
    triples = [(menu, model, seed) for menu, model in pairs for seed in config['final_runs']['seeds']]
    records = {}
    for menu, model, seed in triples:
        path = ctx.dirs.runs / 'summary' / f'{triple_id(menu, model, seed)}.json'
        assert path.exists(), f'{path} is missing: finish step 04 first'
        r = json.loads(path.read_text(encoding='utf-8'))
        assert r['shared_code_version'] == SHARED_CODE_VERSION, (
            f'{path.name} was written with shared code version {r["shared_code_version"]}, this notebook has '
            f'{SHARED_CODE_VERSION}')
        records[(menu, model, seed)] = r
    ctx.records.update(records)
    expected = {t: expected_work(config, t[0], t[1], ctx.settings) for t in triples}
    done = [t for t in triples if (ctx.dirs.out / 'done' / f'{triple_id(*t)}.json').exists()]
    return triples, Progress(expected, done, ctx.dirs.out / 'logs' / f'06-importance-{NOTEBOOK}.log')


def run_triples(ctx, triples, progress):
    for k, (menu, model, seed) in enumerate(triples, 1):
        if (menu, model, seed) in progress.done:
            continue
        t0 = time.time()
        out = explain_triple(ctx, menu, model, seed)
        progress.finish((menu, model, seed))
        parts = ', '.join(f'{name} {fmt_time(s)}' for name, s in out['timings'].items() if name in ('permutation', 'attribution'))
        progress.print(f'[triple {k}/{len(triples)} · {model} × {menu} · seed {seed}] {parts or "checks only"} · '
                       f'{fmt_time(time.time() - t0)} · {progress.status()}')

## 2. Load the inputs and the explain sample

`config.json` (step 02) gives menus and families; `04-final-runs/` the models, their summaries and their saved scores. The explain sample is written by the first of the two computing notebooks that runs; the other one rebuilds it and stops if it differs.

In [8]:
DIRS = SimpleNamespace(runs=RUNS_DIR, out=OUT_DIR)
start = time.time()
data = load_inputs(LOCAL_INPUT)
print(f'Inputs loaded in {fmt_time(time.time() - start)}: {len(data.dataset):,} rows, config version {data.config["version"]}.')

if FORCE_RECOMPUTE:
    for menu, model in data.config['pairs']:
        if model in MODELS:
            for folder, suffix in [('done', '.json'), ('permutation', '.csv'), ('shap_summary', '.csv'), ('shap', '.parquet')]:
                for path in (OUT_DIR / folder).glob(f'{menu}__{model}__seed*{suffix}'):
                    path.unlink()
SAMPLE = load_or_build_sample(data, SETTINGS['sample'], OUT_DIR / 'explain_sample.csv')
print(f"Explain sample: {len(SAMPLE):,} rows ({int(SAMPLE['y'].sum())} illicit, {int((SAMPLE['y'] == 0).sum())} licit, "
      f"licit weight {SAMPLE.loc[SAMPLE['y'] == 0, 'weight'].iloc[0]:.3f}), GNNExplainer core {int(SAMPLE['gnnexplainer_core'].sum())}.")
ctx = SimpleNamespace(data=data, dirs=DIRS, settings=SETTINGS, sample=SAMPLE, records={})
TRIPLES, PROGRESS = plan_triples(ctx, MODELS, ONLY_PAIRS)
PROGRESS.print(f'{len(TRIPLES)} triples in this notebook, {len(PROGRESS.done)} already done.')

Inputs loaded in 0.8s: 203,769 rows, config version 1.
Explain sample: 1,524 rows (524 illicit, 1000 licit, licit weight 9.342), GNNExplainer core 80.
36 triples in this notebook, 0 already done.


## 3. Tests on toy problems

Quick checks with known answers, before the real models:
- **sample**: the 60 illicit of the GNNExplainer core split over time steps 41–49 as 13, 27, 3, 3, 1, 0, 3, 4, 6; the sample rebuilt is identical; the weights add up to the 9,866 test transactions;
- **shuffles**: values move only within their time step; the same seeds give the same permutations;
- **permutation importance**: on a model with known weights, shuffling the ignored column changes nothing (drop exactly 0), shuffling the decisive one makes the PR-AUC collapse;
- **summaries**: shares add up to 1, the density of a family is its share over its share of columns;
- **attributions**: on a Logistic Regression, base + $\sum_j\phi_j$ = the logit; on a Random Forest, base + $\sum_j\phi_j$ = the probability (TreeSHAP is exact).

In [9]:
# The explain sample and the shuffles.
counts = largest_remainder(60, [116, 239, 24, 24, 5, 2, 22, 36, 56])      # the illicit of time steps 41-49
assert counts.tolist() == [13, 27, 3, 3, 1, 0, 3, 4, 6], counts
again = build_explain_sample(data, SETTINGS['sample'])
assert np.array_equal(again['row'], SAMPLE['row']) and np.allclose(again['weight'], SAMPLE['weight'])
assert abs(SAMPLE['weight'].sum() - len(eval_rows(data, 'test'))) < 1e-6
core = SAMPLE[SAMPLE['gnnexplainer_core']]
assert (core['y'] == 1).sum() == SETTINGS['sample']['core_illicit'] and (core['y'] == 0).sum() == SETTINGS['sample']['core_licit']
sub = sample_subset(SAMPLE, 200)
assert len(sub) == int(SAMPLE['y'].sum()) + 200 and abs(sub['weight'].sum() - SAMPLE['weight'].sum()) < 1e-6

steps_toy = np.array([1, 1, 1, 2, 2, 3, 3, 3, 3])
values_toy = np.arange(9) * 10
perms_toy = within_step_permutations(steps_toy, 3)
for p in perms_toy:
    assert all(sorted(values_toy[p][steps_toy == t]) == sorted(values_toy[steps_toy == t]) for t in (1, 2, 3))
assert all(np.array_equal(a, b) for a, b in zip(perms_toy, within_step_permutations(steps_toy, 3)))
assert not np.array_equal(perms_toy[0], perms_toy[1])

# Permutation importance on a model with known weights: column 0 decides, column 1 is ignored.
rng = np.random.default_rng(0)
X_toy = rng.standard_normal((2000, 2))
y_toy = (X_toy[:, 0] > 1).astype(int)
score_toy = lambda X: sigmoid64(4 * X[:, 0] + 0 * X[:, 1])
units_toy = {'decisive': ('column', 'A', [0]), 'ignored': ('column', 'A', [1]), 'both': ('family', 'A', [0, 1])}
table_toy, restored = permutation_importance(score_toy, X_toy, y_toy, units_toy, within_step_permutations(np.zeros(2000), 5))
drops = table_toy.set_index('name')['ap_drop']
assert drops['ignored'] == 0 and drops['decisive'] > 0.5 and abs(drops['both'] - drops['decisive']) < 0.05, drops
assert restored == 0

# Summary of attributions: shares add up to 1, the density of a family is its share over its share of columns.
phi_toy = np.column_stack([np.ones(10), 2 * np.ones(10), np.zeros(10)])
s = attribution_summary(phi_toy, phi_toy, np.r_[np.ones(5), np.zeros(5)], np.ones(10), ['a', 'b', 'c'], ['A', 'A', 'B'],
                        {'A': [0, 1], 'B': [2]}).set_index('name')
assert abs(s.loc[['a', 'b', 'c'], 'share'].sum() - 1) < 1e-12 and abs(s.loc['A', 'share'] - 1) < 1e-12
assert abs(s.loc['A', 'density'] - 1.5) < 1e-12 and s.loc['all', 'mean_abs'] == 3
print('Toy tests passed: sample, shuffles within time steps, permutation importance, summaries.')

Toy tests passed: sample, shuffles within time steps, permutation importance, summaries.


In [10]:
# Attributions of the two models of this notebook on a toy table.
rng = np.random.default_rng(1)
X_toy = rng.standard_normal((600, 3)).astype(np.float32)
y_toy = (X_toy[:, 0] + 0.5 * X_toy[:, 1] + 0.3 * rng.standard_normal(600) > 1).astype(int)
lr_toy = LogisticRegression(max_iter=1000).fit(X_toy, y_toy)
phi, base = linear_shap(lr_toy, X_toy, X_toy.astype(np.float64).mean(axis=0))
assert np.allclose(base + phi.sum(axis=1), lr_toy.decision_function(X_toy), atol=1e-6)
rf_toy = RandomForestClassifier(n_estimators=30, random_state=0).fit(X_toy, y_toy)
phi, base = tree_shap_forest(rf_toy, X_toy[:50])
assert np.allclose(base + phi.sum(axis=1), rf_toy.predict_proba(X_toy[:50])[:, 1], atol=1e-6)
assert np.abs(phi[:, 0]).mean() > np.abs(phi[:, 2]).mean()     # the decisive column weighs more than the noise
print('Toy tests passed: linear SHAP and TreeSHAP add up to the output of the model.')

Toy tests passed: linear SHAP and TreeSHAP add up to the output of the model.


## 4. Timing

Measured on the largest Random Forest of this notebook (seed 0, the deepest trees): one prediction of the test, and TreeSHAP on 20 rows. TreeSHAP is exact but its cost grows with the number of leaves and the depth of the trees, and the forests of `local_agg` and `local_agg_graph` have trees without a depth limit. So, as for Integrated Gradients on the GNNs (decision F8), with `'auto'`:
- if TreeSHAP on the whole explain sample (524 + 1,000 rows) for every seed fits in `rf_shap_budget_hours`, it runs on all three seeds;
- otherwise it runs on seed 0 only, with as many licit rows as fit in the budget (at least 200; the 524 illicit are always kept, and the weights are adjusted).

The choice is saved in `settings-cpu.json` the first time, so that every session uses the same one. Then the estimate of the total time.

In [11]:
rf_pairs = [(m, mo) for m, mo in dict.fromkeys((t[0], t[1]) for t in TRIPLES) if mo == 'rf']
if rf_pairs:
    menu = max(rf_pairs, key=lambda p: len(data.config['menus'][p[0]]['columns']))[0]
    record = ctx.records[(menu, 'rf', 0)]
    forest = load_classical('rf', RUNS_DIR / record['files']['model'])
    X_t = feature_matrix(data, menu, 'raw')[eval_rows(data, 'test')]
    t0 = time.perf_counter()
    forest.predict_proba(X_t)
    t_predict = time.perf_counter() - t0
    t0 = time.perf_counter()
    tree_shap_forest(forest, X_t[:20])
    t_row = (time.perf_counter() - t0) / 20
    del forest
    rf_triples = [t for t in TRIPLES if t[1] == 'rf']
    budget = SETTINGS['rf_shap_budget_hours'] * 3600
    settings_file = OUT_DIR / 'settings-cpu.json'
    if settings_file.exists():
        saved_settings = json.loads(settings_file.read_text(encoding='utf-8'))
        SETTINGS['rf_shap_seeds'], SETTINGS['rf_licit'] = saved_settings['rf_shap_seeds'], saved_settings['rf_licit']
        print(f'Choices of an earlier session ({settings_file.name}) kept.')
    else:
        if SETTINGS['rf_shap_seeds'] == 'auto':
            full = len(rf_triples) * (524 + 1000) * t_row
            SETTINGS['rf_shap_seeds'] = [0, 1, 2] if full <= budget else [0]
        if SETTINGS['rf_licit'] == 'auto':
            n_shap = len([t for t in rf_triples if t[2] in SETTINGS['rf_shap_seeds']])
            SETTINGS['rf_licit'] = int(np.clip(budget / max(n_shap, 1) / t_row - 524, 200, 1000))
        write_json({'rf_shap_seeds': SETTINGS['rf_shap_seeds'], 'rf_licit': SETTINGS['rf_licit'],
                    'seconds_per_row': t_row, 'seconds_per_prediction': t_predict}, settings_file)
    left = [t for t in rf_triples if t not in PROGRESS.done]
    units = sum(len(menu_units(data.config, m)[0]) + len(menu_units(data.config, m, True)[0]) for m, mo, s in left)
    n_shap_left = len([t for t in left if t[2] in SETTINGS['rf_shap_seeds']])
    estimate = units * SETTINGS['shuffles'] * t_predict + n_shap_left * (524 + SETTINGS['rf_licit']) * t_row
    print(f'Random Forest on {menu}: one prediction of the test {t_predict:.2f}s, TreeSHAP {t_row:.2f}s per row.')
    print(f"TreeSHAP of the Random Forest: seeds {SETTINGS['rf_shap_seeds']}, on all 524 illicit + "
          f"{SETTINGS['rf_licit']} licit rows of the sample.")
    print(f'Upper estimate of the {len(left)} Random Forest triples left (every forest as slow as this one): '
          f'~{fmt_time(estimate)}. Logistic Regression: minutes.')
else:
    SETTINGS['rf_shap_seeds'] = [0, 1, 2] if SETTINGS['rf_shap_seeds'] == 'auto' else SETTINGS['rf_shap_seeds']
    SETTINGS['rf_licit'] = 1000 if SETTINGS['rf_licit'] == 'auto' else SETTINGS['rf_licit']
for key in PROGRESS.expected:
    PROGRESS.expected[key] = expected_work(data.config, key[0], key[1], SETTINGS)

Random Forest on local_agg_graph: one prediction of the test 0.39s, TreeSHAP 0.21s per row.
TreeSHAP of the Random Forest: seeds [0, 1, 2], on all 524 illicit + 1000 licit rows of the sample.
Upper estimate of the 18 Random Forest triples left (every forest as slow as this one): ~2h 26m. Logistic Regression: minutes.


## 5. Run

One log line per triple, for example

```
[triple 20/36 · rf × transparent · seed 1] permutation 9m 40s, attribution 6m 05s · 15m 50s · done 20, left 16 · elapsed 3h 10m, left ~2h 05m
```

Before explaining a triple, the reloaded model must give back the saved scores of step 04 exactly (otherwise the notebook stops): the explanations are about the models of the results.

In [12]:
run_triples(ctx, TRIPLES, PROGRESS)

[triple 1/36 · logreg × local · seed 0] permutation 7.6s, attribution 0.0s · 12.0s · done 1, left 35 · elapsed 22.2s, left ~18h 53m
[triple 2/36 · logreg × local · seed 1] permutation 9.5s, attribution 0.0s · 13.4s · done 2, left 34 · elapsed 35.7s, left ~15h 09m
[triple 3/36 · logreg × local · seed 2] permutation 10.4s, attribution 0.0s · 13.2s · done 3, left 33 · elapsed 48.9s, left ~13h 51m
[triple 4/36 · logreg × local_agg · seed 0] permutation 18.4s, attribution 0.1s · 22.2s · done 4, left 32 · elapsed 1m 11s, left ~12h 57m
[triple 5/36 · logreg × local_agg · seed 1] permutation 19.4s, attribution 0.1s · 23.0s · done 5, left 31 · elapsed 1m 34s, left ~12h 38m
[triple 6/36 · logreg × local_agg · seed 2] permutation 18.1s, attribution 0.1s · 21.5s · done 6, left 30 · elapsed 1m 55s, left ~12h 17m
[triple 7/36 · logreg × graph · seed 0] permutation 1.7s, attribution 0.0s · 4.2s · done 7, left 29 · elapsed 1m 59s, left ~12h 03m
[triple 8/36 · logreg × graph · seed 1] permutation 1.7s,

## 6. Summary

Triples finished, and the checks of this notebook (every row of `checks.csv` has a limit; the notebook would have stopped at the first failure).

In [13]:
mine = [triple_id(*t) for t in TRIPLES]
done = [t for t in mine if (OUT_DIR / 'done' / f'{t}.json').exists()]
print(f'Triples of this notebook finished: {len(done)} of {len(mine)}.')
assert len(done) == len(mine), 'some triples are missing: run the notebook again'
checks = pd.read_csv(OUT_DIR / 'checks.csv')
checks = checks[checks['triple'].isin(mine)]
display(checks.groupby('check').agg(triples=('triple', 'count'), largest=('value', 'max'), limit=('limit', 'first'),
                                    all_ok=('ok', 'all')))
drive.flush_and_unmount()
print('Every file is on Drive in', OUT_DIR)

Triples of this notebook finished: 36 of 36.


,triples,largest,limit,all_ok
check,,,,
TreeSHAP: base + sum of phi = probability,18,4.213976e-08,1.000000e-04,True
linear SHAP: base + sum of phi = logit,18,2.273737e-13,1.000000e-06,True
permutation: scores restored,36,3.330669e-16,1.000000e-09,True
reload: largest score difference,36,3.330669e-16,1.000000e-12,True


Every file is on Drive in /content/drive/MyDrive/UNIVERSITA/block-crypto/data/06-feature-importance
